# FINAL INTEGRATED STRATEGY PERFORMANCE EVALUATION

The culminating evaluation of the `weekly_sector_rotation` research project (R1 through R10B accepted). **Not a component-discovery or parameter-search stage**: no new features, targets, architectures, K values, thresholds, or risk rules; no tuning toward any reported CAGR. Assembles the 24 final candidate variants (2 models x 2 K values x 2 allocation methods x 3 risk configurations -- RISK_NONE reproducing R10A, RISK_FULL_A_B_C_D_E reproducing R10B's full stack, and RISK_A_C_D_E, the evidence-based removal of Rule B per R10B's finding that Rule B was individually harmful), verified to reproduce R10A and R10B exactly before any new diagnostics are computed. Reports gross and 10bps-net performance, break-even transaction cost, a diagnostic cost-robustness curve, full performance/risk metrics, weekly alpha/excess-return statistics with block-bootstrap 95% CIs, year-by-year results, predeclared stress periods, a predeclared temporal subperiod split, performance-concentration analysis, leave-one-year-out sensitivity (diagnostic only), the model/K/allocation/risk-configuration comparisons (including an explicit Rule B final diagnostic), a final candidate ranking, and a final alpha/deployability classification. Does **not** begin live trading integration. See `docs/weekly_sector_rotation_final_performance.md` for the full write-up and `final_strategy_evaluation.py` for the shared, unit-tested evaluation-layer functions this notebook imports.

In [1]:
"""
FINAL INTEGRATED STRATEGY PERFORMANCE EVALUATION (scratchpad validation run).
Once verified end-to-end, transcribed into
notebooks/final_integrated_strategy_performance.ipynb and executed there for
the committed artifacts. Not a component-discovery stage: no new features,
targets, architectures, K values, thresholds, or risk rules -- everything
through the simulation engine (data load, NN v1/v2 training, SPY benchmark,
Top-K ranking, RiskFilterEngine) is reused verbatim from
R1/R7/R8/R9/R10A/R10B. The only new logic is the three named risk
configurations (RISK_NONE, RISK_FULL_A_B_C_D_E, RISK_A_C_D_E -- the last
uses the existing RiskFilterEngine with a different active_filters set, not
a new filter) and the final-stage evaluation/reporting layer in
final_strategy_evaluation.py (break-even cost, cost-robustness curve,
temporal thirds, concentration, leave-one-year-out).
"""
"""
FINAL INTEGRATED STRATEGY PERFORMANCE EVALUATION (scratchpad validation run,
part 1: data/model/benchmark/simulation setup + R10A/R10B reproduction
verification). No new features, targets, architectures, K values, or risk
rules -- everything through the simulation engine is reused verbatim from
R1/R7/R8/R9/R10A/R10B.
"""
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

t0 = time.time()
REPO_ROOT = Path("/Users/gavinhussey/Downloads/AtlasQuant")
FEATURE_PANEL_PATH = REPO_ROOT / "data" / "processed" / "weekly_sector_rotation" / "feature_panel.csv"
PRICES_ROOT = REPO_ROOT / "data" / "raw" / "weekly_sector_rotation" / "prices"
OUTPUTS_ROOT = REPO_ROOT / "research" / "strategies" / "weekly_sector_rotation" / "outputs"

sys.path.insert(0, str(REPO_ROOT / "research" / "strategies" / "weekly_sector_rotation"))
import r7_financial_losses as r7  # noqa: E402
import r10a_portfolio as r10a  # noqa: E402
import r10b_risk_filters as r10b  # noqa: E402
import final_strategy_evaluation as fin  # noqa: E402

SECTOR_ETFS = r7.SECTOR_ETFS
BENCHMARK_ETF = r7.BENCHMARK_ETF
FULL_FEATURE_COLS = r7.FULL_FEATURE_COLS

# ---------------------------------------------------------------------------

## 1. Load panel + R1 raw execution prices, authoritative feature set, fold schedule -- verbatim R1/R7/R8/R9/R10A/R10B.

In [2]:
# 1. Load panel + R1 raw execution prices, authoritative feature set, fold
#    schedule -- verbatim R1/R7/R8/R9/R10A/R10B.
# ---------------------------------------------------------------------------
date_cols = ["date", "week_end", "actual_last_trading_date", "feature_cutoff_timestamp",
             "next_week_first_trading_date", "next_week_last_trading_date"]
panel = pd.read_csv(FEATURE_PANEL_PATH, parse_dates=date_cols)
panel = panel.sort_values(["symbol", "date"]).reset_index(drop=True)
AUDIT_COLUMNS = ["week_end", "actual_last_trading_date", "week_complete", "feature_cutoff_timestamp",
                  "next_week_first_trading_date", "next_week_last_trading_date"]
FEATURE_COLUMNS = [c for c in panel.columns if c not in {"date", "symbol", *AUDIT_COLUMNS}]

r1_targets = pd.read_csv(OUTPUTS_ROOT / "r1_target_definitions_full_panel.csv", parse_dates=["date"])
panel = panel.merge(
    r1_targets[["date", "symbol", "target_abs_1pct_next_week", "next_week_first_open",
                "next_week_final_close", "next_week_open_to_close_return", "next_week_spy_return"]],
    on=["date", "symbol"], how="left",
)
assert set(FULL_FEATURE_COLS).issubset(set(FEATURE_COLUMNS))

md = panel.dropna(subset=FEATURE_COLUMNS + ["target_abs_1pct_next_week"]).copy()
md["label_binary"] = md["target_abs_1pct_next_week"].astype(int)
md["year"] = md["date"].dt.year
years_all = sorted(md["year"].unique())
assert years_all == list(range(2005, 2027))
test_years = r7.derive_r1_fold_schedule(years_all)
assert test_years == list(range(2008, 2027)), f"fold schedule diverges from validated R1 schedule: {test_years}"
print(f"[{time.time()-t0:.0f}s] data loaded, {len(md)} rows, test_years={test_years[0]}-{test_years[-1]}")

# ---------------------------------------------------------------------------

[0s] data loaded, 12253 rows, test_years=2008-2026


## 2. SPY execution-aligned benchmark -- verbatim R10A/R10B.

In [3]:
# 2. SPY execution-aligned benchmark -- verbatim R10A/R10B.
# ---------------------------------------------------------------------------
week_bounds = md[["date", "next_week_first_trading_date", "next_week_last_trading_date"]].drop_duplicates(subset=["date"])
spy_daily = pd.read_csv(PRICES_ROOT / f"{BENCHMARK_ETF}.csv", parse_dates=["date"])[["date", "open", "close"]]
spy_open_lookup = spy_daily.set_index("date")["open"]
spy_close_lookup = spy_daily.set_index("date")["close"]
spy_bench_rows = []
for _, row in week_bounds.iterrows():
    entry_date, exit_date = row["next_week_first_trading_date"], row["next_week_last_trading_date"]
    if pd.isna(entry_date) or pd.isna(exit_date):
        continue
    if entry_date not in spy_open_lookup.index or exit_date not in spy_close_lookup.index:
        continue
    entry_open = spy_open_lookup.loc[entry_date]; exit_close = spy_close_lookup.loc[exit_date]
    spy_bench_rows.append({"date": row["date"], "spy_trade_return": r10a.trade_return(entry_open, exit_close)})
spy_benchmark = pd.DataFrame(spy_bench_rows).set_index("date")
spy_weekly_for_metrics = spy_benchmark.reset_index().rename(columns={"spy_trade_return": "portfolio_return"})
spy_weekly_for_metrics["equity_after"] = r10a.equity_curve_from_returns(
    spy_weekly_for_metrics["portfolio_return"].to_numpy())[1:]
print(f"[{time.time()-t0:.0f}s] Step 2 (SPY benchmark) done, {len(spy_benchmark)} weeks.")

# ---------------------------------------------------------------------------

[0s] Step 2 (SPY benchmark) done, 1114 weeks.


## 3. Model definitions + walk-forward training -- verbatim R1/R7/R8/R9/R10A/R10B -- then verify deterministic pooled AUC reproduces the accepted values exactly. No architecture/hyperparameter changes of any kind.

In [4]:
# 3. Model definitions + walk-forward training -- verbatim R1/R7/R8/R9/R10A/R10B.
# ---------------------------------------------------------------------------
NN_SEED = 20260812
HIDDEN_1, HIDDEN_2 = 16, 8
DROPOUT = 0.3
WEIGHT_DECAY = 1e-4
LEARNING_RATE = 1e-3
MAX_EPOCHS = 300
PATIENCE = 15


class SmallMLP(nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, HIDDEN_1), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_1, HIDDEN_2), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


V2_HIDDEN = 32
V2_DROPOUT = 0.4
V2_WEIGHT_DECAY = 1e-3
V2_LEARNING_RATE = 1e-3
V2_MAX_EPOCHS = 300
V2_PATIENCE = 15
N_SEEDS = 5
SEED_BASE = 20260812


class WideSingleLayerMLP(nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, V2_HIDDEN), nn.ReLU(), nn.Dropout(V2_DROPOUT),
            nn.Linear(V2_HIDDEN, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


BCE_LOSS = nn.BCEWithLogitsLoss()


def train_one_model(X_fit, y_fit, X_val, y_val, n_features, seed, v2=False):
    torch.manual_seed(seed)
    model = WideSingleLayerMLP(n_features) if v2 else SmallMLP(n_features)
    lr = V2_LEARNING_RATE if v2 else LEARNING_RATE
    wd = V2_WEIGHT_DECAY if v2 else WEIGHT_DECAY
    max_epochs = V2_MAX_EPOCHS if v2 else MAX_EPOCHS
    patience = V2_PATIENCE if v2 else PATIENCE
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    X_fit_t = torch.tensor(X_fit, dtype=torch.float32); y_fit_t = torch.tensor(y_fit, dtype=torch.float32)
    X_val_t = torch.tensor(X_val, dtype=torch.float32); y_val_t = torch.tensor(y_val, dtype=torch.float32)
    best_val_loss = float("inf"); best_state = None; epochs_without_improvement = 0
    for _ in range(max_epochs):
        model.train(); optimizer.zero_grad()
        loss = BCE_LOSS(model(X_fit_t), y_fit_t)
        loss.backward(); optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = BCE_LOSS(model(X_val_t), y_val_t).item()
        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                break
    model.load_state_dict(best_state); model.eval()
    return model


def walk_forward_bce(md, years_, test_years_, feature_cols, v2=False):
    oof_frames = []
    for test_year in test_years_:
        train_years_available = [y for y in years_ if y < test_year]
        val_year = train_years_available[-1]
        fit_years = train_years_available[:-1]
        train_mask = md["year"].isin(train_years_available)
        fit_mask = md["year"].isin(fit_years)
        val_mask = md["year"] == val_year
        test_mask = md["year"] == test_year
        scaler = StandardScaler().fit(md.loc[train_mask, feature_cols])
        X_fit = scaler.transform(md.loc[fit_mask, feature_cols])
        X_val = scaler.transform(md.loc[val_mask, feature_cols])
        X_test = scaler.transform(md.loc[test_mask, feature_cols])
        y_fit = md.loc[fit_mask, "label_binary"].to_numpy()
        y_val = md.loc[val_mask, "label_binary"].to_numpy()

        if v2:
            seed_predictions = []
            for seed_offset in range(N_SEEDS):
                model = train_one_model(X_fit, y_fit, X_val, y_val, len(feature_cols), SEED_BASE + seed_offset, v2=True)
                with torch.no_grad():
                    seed_predictions.append(torch.sigmoid(model(torch.tensor(X_test, dtype=torch.float32))).numpy())
            predicted_proba = np.mean(seed_predictions, axis=0)
        else:
            model = train_one_model(X_fit, y_fit, X_val, y_val, len(feature_cols), NN_SEED, v2=False)
            with torch.no_grad():
                predicted_proba = torch.sigmoid(model(torch.tensor(X_test, dtype=torch.float32))).numpy()

        fold = md.loc[test_mask, ["date", "symbol", "next_week_first_open", "next_week_final_close",
                                   "next_week_open_to_close_return", "next_week_spy_return", "label_binary"]].copy()
        fold["predicted_proba"] = predicted_proba
        oof_frames.append(fold)
    return pd.concat(oof_frames, ignore_index=True)


oof_v1 = walk_forward_bce(md, years_all, test_years, FULL_FEATURE_COLS, v2=False)
auc_v1 = roc_auc_score(oof_v1.label_binary, oof_v1.predicted_proba)
print(f"[{time.time()-t0:.0f}s] NN_v1 deterministic pooled_auc={auc_v1:.6f} (expected 0.568010)")
assert abs(auc_v1 - 0.5680100994943317) < 1e-4, "NN v1 AUC does not reproduce accepted value"

oof_v2 = walk_forward_bce(md, years_all, test_years, FULL_FEATURE_COLS, v2=True)
auc_v2 = roc_auc_score(oof_v2.label_binary, oof_v2.predicted_proba)
print(f"[{time.time()-t0:.0f}s] NN_v2 deterministic pooled_auc={auc_v2:.6f} (expected 0.573573)")
assert abs(auc_v2 - 0.5735730758971164) < 1e-4, "NN v2 AUC does not reproduce accepted value"
print(f"[{time.time()-t0:.0f}s] Step 3 (walk-forward training, both models) done.")

# ---------------------------------------------------------------------------

[4s] NN_v1 deterministic pooled_auc=0.568010 (expected 0.568010)


[19s] NN_v2 deterministic pooled_auc=0.573573 (expected 0.573573)
[19s] Step 3 (walk-forward training, both models) done.


## 4. Build the FULL 11-sector weekly ranking table per model -- verbatim R10B.

In [5]:
# 4. Build FULL 11-sector weekly ranking table per model -- verbatim R10B.
# ---------------------------------------------------------------------------
OOF_BY_MODEL = {"NN_v1": oof_v1, "NN_v2": oof_v2}


def build_full_ranking(oof: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for date, g in oof.groupby("date"):
        if len(g) < 11:
            continue
        ranked = g["predicted_proba"].rank(ascending=False, method="first")
        gg = g.copy(); gg["rank"] = ranked.astype(int).to_numpy()
        rows.append(gg)
    return pd.concat(rows, ignore_index=True).sort_values(["date", "rank"]).reset_index(drop=True)


full_ranking = {model_name: build_full_ranking(oof) for model_name, oof in OOF_BY_MODEL.items()}
print(f"[{time.time()-t0:.0f}s] Step 4 (full 11-sector weekly ranking, both models) done.")

# ---------------------------------------------------------------------------

[19s] Step 4 (full 11-sector weekly ranking, both models) done.


## 5. Simulation engine -- verbatim R10B (identical trade/halt/allocation logic; only the risk-configuration frozenset passed in differs across the 24-variant grid).

In [6]:
# 5. Simulation engine -- verbatim R10B.
# ---------------------------------------------------------------------------
def simulate(full_rank: pd.DataFrame, k: int, allocation_method: str, active_filters: frozenset):
    engine = r10b.RiskFilterEngine(symbols=SECTOR_ETFS, active_filters=active_filters)
    alloc_tracker = r10a.AllocationStateTracker(SECTOR_ETFS)
    dates_sorted = sorted(full_rank["date"].unique())

    ledger_rows = []
    equity_running = r10a.STARTING_CAPITAL

    for week_idx, date in enumerate(dates_sorted):
        week = full_rank[full_rank["date"] == date].sort_values("rank")
        month = date.month
        year = date.year
        ranked_symbols = week["symbol"].tolist()

        halted = engine.is_halted(week_idx)
        eligible = set() if halted else engine.eligible_symbols(week_idx, month)
        selected = [] if halted else engine.select_top_k_from_eligible(ranked_symbols, eligible, k)

        if selected:
            if allocation_method == "EQUAL_WEIGHT":
                weights = {s: r10a.equal_weight(len(selected)) for s in selected}
            else:
                raw = {s: alloc_tracker.raw_weight_for(s) for s in selected}
                weights = r10a.normalize_paper_weights(raw)
        else:
            weights = {}

        trades = {}
        gross_portfolio_return = 0.0
        for _, r in week[week["symbol"].isin(selected)].iterrows():
            s = r["symbol"]; w = weights[s]
            gross_ret = r["next_week_open_to_close_return"]
            trades[s] = gross_ret
            gross_portfolio_return += w * gross_ret
            ledger_rows.append({
                "date": date, "symbol": s, "rank": int(r["rank"]), "weight": w,
                "entry_open": r["next_week_first_open"], "exit_close": r["next_week_final_close"],
                "gross_trade_return": gross_ret, "cost_adjusted_trade_return": r10a.apply_round_trip_cost(gross_ret),
                "spy_trade_return": r["next_week_spy_return"], "n_selected_this_week": len(selected),
                "halted": halted,
            })

        equity_before = equity_running
        equity_after = equity_before * (1 + gross_portfolio_return)
        portfolio_dollar_pnl = equity_after - equity_before

        for s in selected:
            alloc_tracker.record_trade(s, trades[s])
        engine.record_week_outcome(week_idx, month, year, trades, portfolio_dollar_pnl, equity_before, equity_after)
        equity_running = equity_after

    ledger = pd.DataFrame(ledger_rows)
    return ledger, dates_sorted, engine


print(f"[{time.time()-t0:.0f}s] Step 5 (simulation engine) defined.")

# ---------------------------------------------------------------------------

[19s] Step 5 (simulation engine) defined.


## 6. CRITICAL CHECKPOINT: run the 24-variant grid (2 models x 2 K x 2 allocations x 3 risk configs), then verify RISK_NONE reproduces R10A exactly and RISK_FULL_A_B_C_D_E reproduces R10B's ABCDE exactly, for all 8 base (model,K,allocation) combinations.

In [7]:
# 6. Run the 24-variant grid (2 models x 2 K x 2 allocations x 3 risk
#    configs), then verify RISK_NONE reproduces R10A exactly and
#    RISK_FULL_A_B_C_D_E reproduces R10B's ABCDE exactly.
# ---------------------------------------------------------------------------
all_ledgers = {}
all_engines = {}
for model_name in fin.MODELS:
    for k in fin.K_VALUES:
        for alloc in fin.ALLOCATION_METHODS:
            for risk_name, filters in fin.RISK_CONFIGS.items():
                ledger, dates_sorted, engine = simulate(full_ranking[model_name], k, alloc, filters)
                all_ledgers[(model_name, k, alloc, risk_name)] = ledger
                all_engines[(model_name, k, alloc, risk_name)] = engine
print(f"[{time.time()-t0:.0f}s] Step 6a: {len(all_ledgers)} simulations run (expected 24).")
assert len(all_ledgers) == 24

r10a_performance = pd.read_csv(OUTPUTS_ROOT / "r10a_performance_summary.csv")
r10b_performance = pd.read_csv(OUTPUTS_ROOT / "r10b_performance_summary.csv")

for model_name in fin.MODELS:
    for k in fin.K_VALUES:
        for alloc in fin.ALLOCATION_METHODS:
            ledger = all_ledgers[(model_name, k, alloc, "RISK_NONE")]
            weekly_ret = ledger.groupby("date").apply(lambda g: (g["weight"] * g["gross_trade_return"]).sum(),
                                                        include_groups=False).rename("portfolio_return").reset_index()
            weekly_ret = weekly_ret.sort_values("date").reset_index(drop=True)
            equity = r10a.equity_curve_from_returns(weekly_ret["portfolio_return"].to_numpy())
            ending_equity = equity[-1]
            r10a_row = r10a_performance[(r10a_performance.model == model_name) & (r10a_performance.K == k) &
                                         (r10a_performance.allocation_method == alloc)].iloc[0]
            diff = abs(ending_equity - r10a_row["ending_capital"])
            assert diff < 1.0, f"RISK_NONE does not reproduce R10A for {model_name} K={k} {alloc}: diff={diff}"
            assert len(ledger) == r10a_row["total_etf_trades"]

            ledger_full = all_ledgers[(model_name, k, alloc, "RISK_FULL_A_B_C_D_E")]
            weekly_full = ledger_full.groupby("date").apply(lambda g: (g["weight"] * g["gross_trade_return"]).sum(),
                                                              include_groups=False).rename("portfolio_return").reset_index()
            weekly_full = weekly_full.sort_values("date").reset_index(drop=True)
            equity_full = r10a.equity_curve_from_returns(weekly_full["portfolio_return"].to_numpy())
            ending_equity_full = equity_full[-1]
            r10b_row = r10b_performance[(r10b_performance.model == model_name) & (r10b_performance.K == k) &
                                         (r10b_performance.allocation_method == alloc) &
                                         (r10b_performance.filter_config == "ABCDE")].iloc[0]
            diff_full = abs(ending_equity_full - r10b_row["gross_ending_capital"])
            assert diff_full < 1.0, f"RISK_FULL does not reproduce R10B ABCDE for {model_name} K={k} {alloc}: diff={diff_full}"
            assert len(ledger_full) == r10b_row["total_etf_trades"]
print(f"[{time.time()-t0:.0f}s] Step 6b: RISK_NONE==R10A and RISK_FULL_A_B_C_D_E==R10B(ABCDE) VERIFIED for all 8 base variants.")
print("PART 1 DONE.")

[28s] Step 6a: 24 simulations run (expected 24).


[29s] Step 6b: RISK_NONE==R10A and RISK_FULL_A_B_C_D_E==R10B(ABCDE) VERIFIED for all 8 base variants.
PART 1 DONE.


## 7. Gross and 10bps-net performance for all 24 final candidates.

In [8]:
CANDIDATES = []
for model_name in fin.MODELS:
    for k in fin.K_VALUES:
        for alloc in fin.ALLOCATION_METHODS:
            for risk_name in fin.RISK_CONFIGS:
                CANDIDATES.append((model_name, k, alloc, risk_name))
assert len(CANDIDATES) == 24

ROUND_TRIP_COST_BPS = r10a.COST_BPS_PER_BUY + r10a.COST_BPS_PER_SELL  # 10bps, R10A convention


def compute_core_metrics(weekly: pd.DataFrame, starting_capital: float = r10a.STARTING_CAPITAL) -> dict:
    returns = weekly["portfolio_return"].to_numpy()
    ending_capital = weekly["equity_after"].iloc[-1]
    years = (pd.Timestamp(weekly["date"].iloc[-1]) - pd.Timestamp(weekly["date"].iloc[0])).days / 365.25
    cumulative_return = ending_capital / starting_capital - 1.0
    cagr_val = r10a.cagr(starting_capital, ending_capital, years)
    mdd = r10a.max_drawdown(returns)
    return {
        "starting_capital": starting_capital, "ending_capital": float(ending_capital),
        "cumulative_return": float(cumulative_return), "cagr": cagr_val,
        "annualized_volatility": r10a.annualized_volatility(returns),
        "sharpe_ratio": r10a.sharpe_ratio(returns), "sortino_ratio": r10a.sortino_ratio(returns),
        "max_drawdown": mdd, "calmar_ratio": r10a.calmar_ratio(cagr_val, mdd),
        "mean_weekly_return": float(np.mean(returns)), "median_weekly_return": float(np.median(returns)),
        "weekly_return_std": float(np.std(returns, ddof=0)),
        "best_week": float(np.max(returns)), "worst_week": float(np.min(returns)),
        "positive_week_rate": float((returns > 0).mean()), "n_weeks": len(weekly),
        "years_elapsed": years,
    }


def settle_from_ledger(ledger: pd.DataFrame, return_col: str, all_dates: list,
                        starting_capital: float = r10a.STARTING_CAPITAL) -> pd.DataFrame:
    if len(ledger) > 0:
        by_week = ledger.groupby("date").apply(lambda g: (g["weight"] * g[return_col]).sum(), include_groups=False)
    else:
        by_week = pd.Series(dtype=float)
    rows = [{"date": d, "portfolio_return": float(by_week.get(d, 0.0))} for d in all_dates]
    weekly = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
    equity = r10a.equity_curve_from_returns(weekly["portfolio_return"].to_numpy(), starting_capital)
    weekly["equity_before"] = equity[:-1]; weekly["equity_after"] = equity[1:]
    return weekly


spy_dates_sorted = sorted(spy_weekly_for_metrics["date"].unique())
spy_core_metrics = compute_core_metrics(spy_weekly_for_metrics)
spy_thirds = fin.chronological_thirds(spy_dates_sorted)

settlements_gross = {}
settlements_net = {}
weekly_excess_store = {}
performance_rows = []

for key in CANDIDATES:
    model_name, k, alloc, risk_name = key
    ledger = all_ledgers[key]
    all_dates = sorted(full_ranking[model_name]["date"].unique())
    weekly_gross = settle_from_ledger(ledger, "gross_trade_return", all_dates)
    weekly_net = settle_from_ledger(ledger, "cost_adjusted_trade_return", all_dates)
    settlements_gross[key] = weekly_gross
    settlements_net[key] = weekly_net

    core_gross = compute_core_metrics(weekly_gross)
    core_net = compute_core_metrics(weekly_net)
    merged = weekly_gross.merge(spy_weekly_for_metrics[["date", "portfolio_return"]], on="date", suffixes=("", "_spy"))
    excess = merged["portfolio_return"] - merged["portfolio_return_spy"]
    weekly_excess_store[key] = merged.assign(excess_return=excess).set_index("date")["excess_return"]

    performance_rows.append({
        "model": model_name, "K": k, "allocation_method": alloc, "risk_config": risk_name,
        "gross_ending_capital": core_gross["ending_capital"], "gross_cagr": core_gross["cagr"],
        "gross_cumulative_return": core_gross["cumulative_return"],
        "gross_sharpe": core_gross["sharpe_ratio"], "gross_sortino": core_gross["sortino_ratio"],
        "gross_max_drawdown": core_gross["max_drawdown"], "gross_calmar": core_gross["calmar_ratio"],
        "gross_annualized_volatility": core_gross["annualized_volatility"],
        "gross_mean_weekly_return": core_gross["mean_weekly_return"],
        "gross_positive_week_rate": core_gross["positive_week_rate"],
        "net_ending_capital": core_net["ending_capital"], "net_cagr": core_net["cagr"],
        "net_cumulative_return": core_net["cumulative_return"],
        "net_sharpe": core_net["sharpe_ratio"], "net_sortino": core_net["sortino_ratio"],
        "net_max_drawdown": core_net["max_drawdown"], "net_calmar": core_net["calmar_ratio"],
        "cagr_drag_from_10bps_costs": core_gross["cagr"] - core_net["cagr"],
        "mean_weekly_excess_return_vs_spy": float(excess.mean()),
        "excess_return_win_rate": float((excess > 0).mean()),
        "total_etf_trades": len(ledger), "n_trade_weeks": ledger["date"].nunique() if len(ledger) else 0,
        "mean_positions_per_week": float(ledger.groupby("date").size().mean()) if len(ledger) else 0.0,
        "n_weeks_total": len(all_dates), "years_elapsed": core_gross["years_elapsed"],
    })
final_performance_summary = pd.DataFrame(performance_rows)
print(f"[{time.time()-t0:.0f}s] Step 7 (gross+net performance, all 24 candidates) done.")
print(final_performance_summary[["model", "K", "allocation_method", "risk_config", "gross_cagr", "net_cagr", "gross_max_drawdown"]].to_string())

# ---------------------------------------------------------------------------

[31s] Step 7 (gross+net performance, all 24 candidates) done.
    model  K        allocation_method          risk_config  gross_cagr  net_cagr  gross_max_drawdown
0   NN_v1  2             EQUAL_WEIGHT            RISK_NONE    0.098547  0.042741           -0.590789
1   NN_v1  2             EQUAL_WEIGHT  RISK_FULL_A_B_C_D_E    0.060742  0.023515           -0.333185
2   NN_v1  2             EQUAL_WEIGHT         RISK_A_C_D_E    0.089364  0.036478           -0.535114
3   NN_v1  2  PAPER_WEIGHT_NORMALIZED            RISK_NONE    0.095003  0.039374           -0.588656
4   NN_v1  2  PAPER_WEIGHT_NORMALIZED  RISK_FULL_A_B_C_D_E    0.062173  0.024896           -0.320979
5   NN_v1  2  PAPER_WEIGHT_NORMALIZED         RISK_A_C_D_E    0.090007  0.036867           -0.542656
6   NN_v1  3             EQUAL_WEIGHT            RISK_NONE    0.080438  0.025538           -0.609950
7   NN_v1  3             EQUAL_WEIGHT  RISK_FULL_A_B_C_D_E    0.064499  0.027200           -0.367543
8   NN_v1  3             EQUA

## 8. Break-even transaction-cost analysis (round-trip bps at which NET CAGR equals SPY CAGR) + the diagnostic-only 0/5/10/15/20bps cost-robustness curve.

In [9]:
# 8. Break-even transaction-cost analysis: round-trip bps at which NET CAGR
#    equals SPY CAGR (the cost level that fully erodes the candidate's edge
#    over the passive benchmark). Diagnostic use of the same 0/5/10/15/20bps
#    cost-robustness curve (also required, separately) -- NOT for tuning.
# ---------------------------------------------------------------------------
break_even_rows = []
cost_curve_rows = []
for key in CANDIDATES:
    model_name, k, alloc, risk_name = key
    weekly_gross = settlements_gross[key]
    gross_returns = weekly_gross["portfolio_return"].to_numpy()
    years_elapsed = performance_rows[CANDIDATES.index(key)]["years_elapsed"]
    be_bps = fin.break_even_round_trip_bps(
        gross_returns, spy_core_metrics["cagr"], r10a.STARTING_CAPITAL, years_elapsed,
        r10a.cagr, r10a.equity_curve_from_returns,
    )
    break_even_rows.append({"model": model_name, "K": k, "allocation_method": alloc, "risk_config": risk_name,
                             "break_even_round_trip_bps_vs_spy": be_bps})
    for bps in fin.COST_ROBUSTNESS_BPS_GRID:
        net_returns = gross_returns - fin.round_trip_cost_fraction_for_bps(bps)
        equity = r10a.equity_curve_from_returns(net_returns, r10a.STARTING_CAPITAL)
        cost_curve_rows.append({
            "model": model_name, "K": k, "allocation_method": alloc, "risk_config": risk_name,
            "round_trip_bps": bps, "cagr_at_cost": r10a.cagr(r10a.STARTING_CAPITAL, equity[-1], years_elapsed),
        })
final_break_even_cost = pd.DataFrame(break_even_rows)
final_cost_robustness_curve = pd.DataFrame(cost_curve_rows)
print(f"[{time.time()-t0:.0f}s] Step 8 (break-even cost + cost-robustness curve) done.")

# ---------------------------------------------------------------------------

[31s] Step 8 (break-even cost + cost-robustness curve) done.


## 9. Weekly alpha / excess-return statistics with block-bootstrap 95% CIs.

In [10]:
# 9. Weekly alpha / excess-return statistics with block-bootstrap 95% CIs.
# ---------------------------------------------------------------------------
alpha_bootstrap_rows = []
for key in CANDIDATES:
    model_name, k, alloc, risk_name = key
    excess = weekly_excess_store[key]
    b = r7.block_bootstrap_ci(excess)
    alpha_bootstrap_rows.append({"model": model_name, "K": k, "allocation_method": alloc, "risk_config": risk_name,
                                  "quantity": "mean_weekly_excess_return_vs_spy", **b})
    gross_ret_series = settlements_gross[key].set_index("date")["portfolio_return"]
    b2 = r7.block_bootstrap_ci(gross_ret_series)
    alpha_bootstrap_rows.append({"model": model_name, "K": k, "allocation_method": alloc, "risk_config": risk_name,
                                  "quantity": "mean_weekly_gross_return_vs_0", **b2})
final_alpha_bootstrap_ci = pd.DataFrame(alpha_bootstrap_rows)
print(f"[{time.time()-t0:.0f}s] Step 9 (weekly alpha/excess bootstrap CIs) done.")

# ---------------------------------------------------------------------------

[39s] Step 9 (weekly alpha/excess bootstrap CIs) done.


## 10. Year-by-year results, all 24 candidates.

In [11]:
# 10. Year-by-year results.
# ---------------------------------------------------------------------------
yearly_rows = []
for key in CANDIDATES:
    model_name, k, alloc, risk_name = key
    w = settlements_gross[key].copy(); w["year"] = pd.to_datetime(w["date"]).dt.year
    spy_y = spy_weekly_for_metrics.copy(); spy_y["year"] = pd.to_datetime(spy_y["date"]).dt.year
    ledger = all_ledgers[key].copy()
    if len(ledger):
        ledger["year"] = pd.to_datetime(ledger["date"]).dt.year
    for year, g in w.groupby("year"):
        spy_g = spy_y[spy_y["year"] == year]
        strat_year_return = float(np.prod(1 + g["portfolio_return"]) - 1)
        spy_year_return = float(np.prod(1 + spy_g["portfolio_return"]) - 1) if len(spy_g) else np.nan
        yearly_rows.append({
            "model": model_name, "K": k, "allocation_method": alloc, "risk_config": risk_name, "year": int(year),
            "strategy_return": strat_year_return, "spy_return": spy_year_return,
            "excess_return": strat_year_return - spy_year_return if not np.isnan(spy_year_return) else np.nan,
            "max_drawdown": r10a.max_drawdown(g["portfolio_return"].to_numpy()),
            "sharpe": r10a.sharpe_ratio(g["portfolio_return"].to_numpy()) if len(g) >= 2 else np.nan,
            "total_etf_trades": int(len(ledger[ledger["year"] == year])) if len(ledger) else 0,
        })
final_yearly_performance = pd.DataFrame(yearly_rows)
print(f"[{time.time()-t0:.0f}s] Step 10 (year-by-year) done.")

# ---------------------------------------------------------------------------

[39s] Step 10 (year-by-year) done.


## 11. Predeclared stress periods (2008 financial crisis, 2020 COVID shock, 2022 bear market).

In [12]:
# 11. Predeclared stress periods (2008, 2020, 2022).
# ---------------------------------------------------------------------------
stress_rows = []
for label, year in fin.STRESS_YEARS.items():
    for key in CANDIDATES:
        model_name, k, alloc, risk_name = key
        row = final_yearly_performance[(final_yearly_performance.model == model_name) & (final_yearly_performance.K == k) &
                                        (final_yearly_performance.allocation_method == alloc) &
                                        (final_yearly_performance.risk_config == risk_name) &
                                        (final_yearly_performance.year == year)]
        if len(row) == 0:
            continue
        r_ = row.iloc[0]
        stress_rows.append({"stress_period": label, "year": year, "model": model_name, "K": k,
                             "allocation_method": alloc, "risk_config": risk_name,
                             "strategy_return": r_["strategy_return"], "spy_return": r_["spy_return"],
                             "excess_return": r_["excess_return"], "max_drawdown": r_["max_drawdown"]})
final_stress_period_metrics = pd.DataFrame(stress_rows)
print(f"[{time.time()-t0:.0f}s] Step 11 (stress periods) done.")

# ---------------------------------------------------------------------------

[39s] Step 11 (stress periods) done.


## 12. Predeclared temporal subperiod split (chronological thirds of the full OOS window).

In [13]:
# 12. Predeclared temporal subperiod split (chronological thirds).
# ---------------------------------------------------------------------------
subperiod_rows = []
for key in CANDIDATES:
    model_name, k, alloc, risk_name = key
    w = settlements_gross[key].set_index("date")
    for sp_name, sp_dates in spy_thirds.items():
        sp_dates_present = [d for d in sp_dates if d in w.index]
        if not sp_dates_present:
            continue
        sub = w.loc[sp_dates_present].sort_index()
        spy_sub = spy_weekly_for_metrics.set_index("date").loc[sp_dates_present].sort_index()
        strat_ret = float(np.prod(1 + sub["portfolio_return"]) - 1)
        spy_ret = float(np.prod(1 + spy_sub["portfolio_return"]) - 1)
        subperiod_rows.append({
            "model": model_name, "K": k, "allocation_method": alloc, "risk_config": risk_name,
            "subperiod": sp_name, "start_date": str(pd.Timestamp(sp_dates_present[0]).date()),
            "end_date": str(pd.Timestamp(sp_dates_present[-1]).date()), "n_weeks": len(sp_dates_present),
            "strategy_return": strat_ret, "spy_return": spy_ret, "excess_return": strat_ret - spy_ret,
            "sharpe": r10a.sharpe_ratio(sub["portfolio_return"].to_numpy()),
            "max_drawdown": r10a.max_drawdown(sub["portfolio_return"].to_numpy()),
        })
final_temporal_subperiods = pd.DataFrame(subperiod_rows)
print(f"[{time.time()-t0:.0f}s] Step 12 (temporal thirds) done.")

# ---------------------------------------------------------------------------

[40s] Step 12 (temporal thirds) done.


## 13. Performance-concentration analysis (year / week / ETF) -- is the track record broad-based or a handful of lucky bets?

In [14]:
# 13. Performance-concentration analysis (year / week / ETF).
# ---------------------------------------------------------------------------
conc_year_frames = []
conc_week_rows = []
conc_etf_frames = []
for key in CANDIDATES:
    model_name, k, alloc, risk_name = key
    by_year = fin.performance_concentration_by_year(settlements_gross[key])
    by_year.insert(0, "model", model_name); by_year.insert(1, "K", k)
    by_year.insert(2, "allocation_method", alloc); by_year.insert(3, "risk_config", risk_name)
    conc_year_frames.append(by_year)

    by_week = fin.performance_concentration_by_week(settlements_gross[key], top_n=10)
    conc_week_rows.append({"model": model_name, "K": k, "allocation_method": alloc, "risk_config": risk_name, **by_week})

    by_etf = fin.performance_concentration_by_etf(all_ledgers[key])
    by_etf.insert(0, "model", model_name); by_etf.insert(1, "K", k)
    by_etf.insert(2, "allocation_method", alloc); by_etf.insert(3, "risk_config", risk_name)
    conc_etf_frames.append(by_etf)
final_concentration_by_year = pd.concat(conc_year_frames, ignore_index=True)
final_concentration_by_week = pd.DataFrame(conc_week_rows)
final_concentration_by_etf = pd.concat(conc_etf_frames, ignore_index=True)
print(f"[{time.time()-t0:.0f}s] Step 13 (performance concentration) done.")

# ---------------------------------------------------------------------------

[40s] Step 13 (performance concentration) done.

## 14. Leave-one-year-out sensitivity (diagnostic only -- NO retraining, NO reselection).

In [15]:
# 14. Leave-one-year-out sensitivity (diagnostic only, no retraining).
# ---------------------------------------------------------------------------
loyo_frames = []
for key in CANDIDATES:
    model_name, k, alloc, risk_name = key
    loyo = fin.leave_one_year_out_cagr(settlements_gross[key], r10a.STARTING_CAPITAL, r10a.cagr, r10a.equity_curve_from_returns)
    loyo.insert(0, "model", model_name); loyo.insert(1, "K", k)
    loyo.insert(2, "allocation_method", alloc); loyo.insert(3, "risk_config", risk_name)
    full_cagr = final_performance_summary[(final_performance_summary.model == model_name) & (final_performance_summary.K == k) &
                                           (final_performance_summary.allocation_method == alloc) &
                                           (final_performance_summary.risk_config == risk_name)]["gross_cagr"].iloc[0]
    loyo["full_period_cagr"] = full_cagr
    loyo["cagr_swing_from_full_period"] = loyo["cagr_excluding_year"] - full_cagr
    loyo_frames.append(loyo)
final_leave_one_year_out = pd.concat(loyo_frames, ignore_index=True)
print(f"[{time.time()-t0:.0f}s] Step 14 (leave-one-year-out) done.")
print("PART 2 DONE.")

# ---------------------------------------------------------------------------

[40s] Step 14 (leave-one-year-out) done.
PART 2 DONE.


## 15. NN v1 vs. NN v2 comparison (descriptive).

In [16]:
# 15. NN v1 vs NN v2 comparison (descriptive, all K/alloc/risk combos).
# ---------------------------------------------------------------------------
v1_vs_v2_rows = []
for k in fin.K_VALUES:
    for alloc in fin.ALLOCATION_METHODS:
        for risk_name in fin.RISK_CONFIGS:
            r1_ = final_performance_summary[(final_performance_summary.model == "NN_v1") & (final_performance_summary.K == k) &
                                             (final_performance_summary.allocation_method == alloc) &
                                             (final_performance_summary.risk_config == risk_name)].iloc[0]
            r2_ = final_performance_summary[(final_performance_summary.model == "NN_v2") & (final_performance_summary.K == k) &
                                             (final_performance_summary.allocation_method == alloc) &
                                             (final_performance_summary.risk_config == risk_name)].iloc[0]
            v1_vs_v2_rows.append({
                "K": k, "allocation_method": alloc, "risk_config": risk_name,
                "nn_v1_gross_cagr": r1_["gross_cagr"], "nn_v2_gross_cagr": r2_["gross_cagr"],
                "nn_v2_minus_nn_v1_gross_cagr": r2_["gross_cagr"] - r1_["gross_cagr"],
                "nn_v1_net_cagr": r1_["net_cagr"], "nn_v2_net_cagr": r2_["net_cagr"],
                "nn_v1_sharpe": r1_["gross_sharpe"], "nn_v2_sharpe": r2_["gross_sharpe"],
                "nn_v1_max_drawdown": r1_["gross_max_drawdown"], "nn_v2_max_drawdown": r2_["gross_max_drawdown"],
            })
final_nn_v1_vs_v2_comparison = pd.DataFrame(v1_vs_v2_rows)

# ---------------------------------------------------------------------------

## 16. Top-2 vs. Top-3 comparison (descriptive, no K optimization).

In [17]:
# 16. Top-2 vs Top-3 comparison (descriptive, no K optimization).
# ---------------------------------------------------------------------------
top2_vs_top3_rows = []
for model_name in fin.MODELS:
    for alloc in fin.ALLOCATION_METHODS:
        for risk_name in fin.RISK_CONFIGS:
            r2_ = final_performance_summary[(final_performance_summary.model == model_name) & (final_performance_summary.K == 2) &
                                             (final_performance_summary.allocation_method == alloc) &
                                             (final_performance_summary.risk_config == risk_name)].iloc[0]
            r3_ = final_performance_summary[(final_performance_summary.model == model_name) & (final_performance_summary.K == 3) &
                                             (final_performance_summary.allocation_method == alloc) &
                                             (final_performance_summary.risk_config == risk_name)].iloc[0]
            top2_vs_top3_rows.append({
                "model": model_name, "allocation_method": alloc, "risk_config": risk_name,
                "top2_gross_cagr": r2_["gross_cagr"], "top3_gross_cagr": r3_["gross_cagr"],
                "top2_net_cagr": r2_["net_cagr"], "top3_net_cagr": r3_["net_cagr"],
                "top2_max_drawdown": r2_["gross_max_drawdown"], "top3_max_drawdown": r3_["gross_max_drawdown"],
                "top2_sharpe": r2_["gross_sharpe"], "top3_sharpe": r3_["gross_sharpe"],
            })
final_top2_vs_top3_comparison = pd.DataFrame(top2_vs_top3_rows)

# ---------------------------------------------------------------------------

## 17. Equal vs. Paper Weight comparison, with block-bootstrap significance.

In [18]:
# 17. Equal vs Paper Weight comparison (with block-bootstrap significance).
# ---------------------------------------------------------------------------
equal_vs_paper_rows = []
for model_name in fin.MODELS:
    for k in fin.K_VALUES:
        for risk_name in fin.RISK_CONFIGS:
            eq_key = (model_name, k, "EQUAL_WEIGHT", risk_name)
            pw_key = (model_name, k, "PAPER_WEIGHT_NORMALIZED", risk_name)
            eq = settlements_gross[eq_key].set_index("date"); pw = settlements_gross[pw_key].set_index("date")
            common = eq.index.intersection(pw.index)
            diff = (pw.loc[common, "portfolio_return"] - eq.loc[common, "portfolio_return"]).sort_index()
            b = r7.block_bootstrap_ci(diff)
            eq_perf = final_performance_summary[(final_performance_summary.model == model_name) & (final_performance_summary.K == k) &
                                                 (final_performance_summary.allocation_method == "EQUAL_WEIGHT") &
                                                 (final_performance_summary.risk_config == risk_name)].iloc[0]
            pw_perf = final_performance_summary[(final_performance_summary.model == model_name) & (final_performance_summary.K == k) &
                                                 (final_performance_summary.allocation_method == "PAPER_WEIGHT_NORMALIZED") &
                                                 (final_performance_summary.risk_config == risk_name)].iloc[0]
            equal_vs_paper_rows.append({
                "model": model_name, "K": k, "risk_config": risk_name,
                "cagr_diff_paper_minus_equal": pw_perf["gross_cagr"] - eq_perf["gross_cagr"],
                "sharpe_diff": pw_perf["gross_sharpe"] - eq_perf["gross_sharpe"],
                "max_drawdown_diff": pw_perf["gross_max_drawdown"] - eq_perf["gross_max_drawdown"],
                "mean_weekly_return_diff_point": b["point_estimate"], "ci_lower": b["ci_lower"],
                "ci_upper": b["ci_upper"], "significant": b["significant"],
            })
final_equal_vs_paper_comparison = pd.DataFrame(equal_vs_paper_rows)

# ---------------------------------------------------------------------------

## 18. Risk-configuration comparison + explicit "Rule B final diagnostic": does evidence-based removal of Rule B (RISK_A_C_D_E) beat the full stack at the final integrated level, confirming R10B's individual-rule finding?

In [19]:
# 18. Risk-configuration comparison + explicit "Rule B final diagnostic":
#     does RISK_A_C_D_E (evidence-based removal of B) beat RISK_FULL_A_B_C_D_E,
#     confirming R10B's individual-rule finding at the final integrated level?
# ---------------------------------------------------------------------------
risk_config_rows = []
rule_b_diagnostic_rows = []
for model_name in fin.MODELS:
    for k in fin.K_VALUES:
        for alloc in fin.ALLOCATION_METHODS:
            none_ = final_performance_summary[(final_performance_summary.model == model_name) & (final_performance_summary.K == k) &
                                               (final_performance_summary.allocation_method == alloc) &
                                               (final_performance_summary.risk_config == "RISK_NONE")].iloc[0]
            full_ = final_performance_summary[(final_performance_summary.model == model_name) & (final_performance_summary.K == k) &
                                               (final_performance_summary.allocation_method == alloc) &
                                               (final_performance_summary.risk_config == "RISK_FULL_A_B_C_D_E")].iloc[0]
            acde_ = final_performance_summary[(final_performance_summary.model == model_name) & (final_performance_summary.K == k) &
                                               (final_performance_summary.allocation_method == alloc) &
                                               (final_performance_summary.risk_config == "RISK_A_C_D_E")].iloc[0]
            for risk_name, row in [("RISK_NONE", none_), ("RISK_FULL_A_B_C_D_E", full_), ("RISK_A_C_D_E", acde_)]:
                risk_config_rows.append({
                    "model": model_name, "K": k, "allocation_method": alloc, "risk_config": risk_name,
                    "gross_cagr": row["gross_cagr"], "net_cagr": row["net_cagr"],
                    "gross_sharpe": row["gross_sharpe"], "gross_max_drawdown": row["gross_max_drawdown"],
                    "gross_calmar": row["gross_calmar"],
                })
            acde_beats_full_cagr = acde_["gross_cagr"] > full_["gross_cagr"]
            acde_beats_full_sharpe = acde_["gross_sharpe"] > full_["gross_sharpe"]
            acde_worse_drawdown = acde_["gross_max_drawdown"] < full_["gross_max_drawdown"]  # more negative = worse
            rule_b_diagnostic_rows.append({
                "model": model_name, "K": k, "allocation_method": alloc,
                "full_stack_gross_cagr": full_["gross_cagr"], "acde_gross_cagr": acde_["gross_cagr"],
                "acde_minus_full_gross_cagr": acde_["gross_cagr"] - full_["gross_cagr"],
                "full_stack_sharpe": full_["gross_sharpe"], "acde_sharpe": acde_["gross_sharpe"],
                "full_stack_max_drawdown": full_["gross_max_drawdown"], "acde_max_drawdown": acde_["gross_max_drawdown"],
                "acde_removing_rule_b_improves_cagr": bool(acde_beats_full_cagr),
                "acde_removing_rule_b_improves_sharpe": bool(acde_beats_full_sharpe),
                "acde_removing_rule_b_worsens_drawdown_protection": bool(acde_worse_drawdown),
                "r10b_individual_rule_b_finding_confirmed_at_integrated_level": bool(acde_beats_full_cagr and acde_beats_full_sharpe),
            })
final_risk_config_comparison = pd.DataFrame(risk_config_rows)
final_rule_b_diagnostic = pd.DataFrame(rule_b_diagnostic_rows)
print(f"[{time.time()-t0:.0f}s] Step 15-18 (model/K/allocation/risk-config comparisons + Rule B diagnostic) done.")
print(final_rule_b_diagnostic[["model", "K", "allocation_method", "acde_minus_full_gross_cagr",
                                "r10b_individual_rule_b_finding_confirmed_at_integrated_level"]].to_string())

# ---------------------------------------------------------------------------

[42s] Step 15-18 (model/K/allocation/risk-config comparisons + Rule B diagnostic) done.
   model  K        allocation_method  acde_minus_full_gross_cagr  r10b_individual_rule_b_finding_confirmed_at_integrated_level
0  NN_v1  2             EQUAL_WEIGHT                    0.028621                                                          True
1  NN_v1  2  PAPER_WEIGHT_NORMALIZED                    0.027834                                                          True
2  NN_v1  3             EQUAL_WEIGHT                    0.009048                                                          True
3  NN_v1  3  PAPER_WEIGHT_NORMALIZED                    0.008946                                                          True
4  NN_v2  2             EQUAL_WEIGHT                    0.037657                                                          True
5  NN_v2  2  PAPER_WEIGHT_NORMALIZED                    0.044652                                                          True
6  NN_v2  3            

## 19. Final candidate ranking -- predeclared criterion (net, cost-aware Sharpe ratio, descending), not tuned to any single candidate's CAGR.

In [20]:
# 19. Final candidate ranking. Ranking criterion (predeclared, not tuned to
#     any single candidate): rank by NET (10bps) Sharpe ratio, descending --
#     the standard risk-adjusted, cost-aware metric; CAGR and max drawdown
#     reported alongside for transparency, not used to break ties beyond
#     Sharpe itself.
# ---------------------------------------------------------------------------
ranking_rows = []
for _, row in final_performance_summary.iterrows():
    ranking_rows.append({
        "model": row["model"], "K": row["K"], "allocation_method": row["allocation_method"],
        "risk_config": row["risk_config"], "net_sharpe": row["net_sharpe"], "net_cagr": row["net_cagr"],
        "gross_cagr": row["gross_cagr"], "gross_max_drawdown": row["gross_max_drawdown"],
        "net_max_drawdown": row["net_max_drawdown"], "gross_calmar": row["gross_calmar"],
    })
final_candidate_ranking = pd.DataFrame(ranking_rows).sort_values("net_sharpe", ascending=False).reset_index(drop=True)
final_candidate_ranking.insert(0, "rank", range(1, len(final_candidate_ranking) + 1))
print(f"[{time.time()-t0:.0f}s] Step 19 (final candidate ranking) done.")
print(final_candidate_ranking.head(10).to_string())

# ---------------------------------------------------------------------------

[42s] Step 19 (final candidate ranking) done.
   rank  model  K        allocation_method          risk_config  net_sharpe  net_cagr  gross_cagr  gross_max_drawdown  net_max_drawdown  gross_calmar
0     1  NN_v2  2             EQUAL_WEIGHT         RISK_A_C_D_E    0.406141  0.062706    0.117146           -0.492873         -0.507737      0.237680
1     2  NN_v2  2  PAPER_WEIGHT_NORMALIZED         RISK_A_C_D_E    0.405575  0.062505    0.116935           -0.496177         -0.510949      0.235671
2     3  NN_v2  2  PAPER_WEIGHT_NORMALIZED            RISK_NONE    0.401679  0.066974    0.124051           -0.543795         -0.562966      0.228121
3     4  NN_v2  2             EQUAL_WEIGHT            RISK_NONE    0.393331  0.065097    0.122076           -0.547693         -0.566704      0.222891
4     5  NN_v2  3  PAPER_WEIGHT_NORMALIZED  RISK_FULL_A_B_C_D_E    0.346887  0.041138    0.077593           -0.361494         -0.415982      0.214647
5     6  NN_v2  3             EQUAL_WEIGHT  RISK_FULL_

## 20. Final alpha classification + deployability classification for the top-ranked candidate, using the full evidentiary picture (bootstrap significance, LOYO stability, cost survivability) -- not CAGR alone. Does NOT begin live trading integration.

In [21]:
# 20. Final alpha classification + deployability classification, for the
#     TOP-RANKED candidate, using the full evidentiary picture (bootstrap
#     significance, LOYO stability, cost survivability, stress-period
#     behavior) -- not CAGR alone.
# ---------------------------------------------------------------------------
top = final_candidate_ranking.iloc[0]
top_key = (top["model"], int(top["K"]), top["allocation_method"], top["risk_config"])
top_perf = final_performance_summary[(final_performance_summary.model == top_key[0]) & (final_performance_summary.K == top_key[1]) &
                                      (final_performance_summary.allocation_method == top_key[2]) &
                                      (final_performance_summary.risk_config == top_key[3])].iloc[0]
top_excess_boot = final_alpha_bootstrap_ci[(final_alpha_bootstrap_ci.model == top_key[0]) & (final_alpha_bootstrap_ci.K == top_key[1]) &
                                            (final_alpha_bootstrap_ci.allocation_method == top_key[2]) &
                                            (final_alpha_bootstrap_ci.risk_config == top_key[3]) &
                                            (final_alpha_bootstrap_ci.quantity == "mean_weekly_excess_return_vs_spy")].iloc[0]
top_loyo = final_leave_one_year_out[(final_leave_one_year_out.model == top_key[0]) & (final_leave_one_year_out.K == top_key[1]) &
                                     (final_leave_one_year_out.allocation_method == top_key[2]) &
                                     (final_leave_one_year_out.risk_config == top_key[3])]
top_break_even = final_break_even_cost[(final_break_even_cost.model == top_key[0]) & (final_break_even_cost.K == top_key[1]) &
                                        (final_break_even_cost.allocation_method == top_key[2]) &
                                        (final_break_even_cost.risk_config == top_key[3])].iloc[0]

excess_significant = bool(top_excess_boot["significant"])
excess_positive = top_excess_boot["point_estimate"] > 0
net_beats_spy = top_perf["net_cagr"] > spy_core_metrics["cagr"]
gross_beats_spy = top_perf["gross_cagr"] > spy_core_metrics["cagr"]
survives_10bps = top_perf["net_cagr"] > 0
loyo_cagr_range = float(top_loyo["cagr_excluding_year"].max() - top_loyo["cagr_excluding_year"].min()) if len(top_loyo) else np.nan
loyo_stable = loyo_cagr_range < 0.05 if not np.isnan(loyo_cagr_range) else False
positive_be = top_break_even["break_even_round_trip_bps_vs_spy"] is not None

if excess_significant and excess_positive and net_beats_spy and loyo_stable:
    alpha_classification = "ROBUST_ALPHA_POSITIVE"
elif excess_positive and gross_beats_spy and (not excess_significant or not loyo_stable):
    alpha_classification = "ALPHA_POSITIVE_BUT_FRAGILE"
elif excess_significant and not excess_positive:
    alpha_classification = "ALPHA_NEGATIVE"
elif not excess_significant and abs(top_excess_boot["point_estimate"]) < 1e-4:
    alpha_classification = "ALPHA_NULL"
else:
    alpha_classification = "ALPHA_MIXED"

if alpha_classification == "ROBUST_ALPHA_POSITIVE" and survives_10bps and positive_be:
    deployability_classification = "PAPER_TRADE_CANDIDATE"
elif alpha_classification in ("ROBUST_ALPHA_POSITIVE", "ALPHA_POSITIVE_BUT_FRAGILE") and survives_10bps:
    deployability_classification = "LIMITED_CAPITAL_PILOT_CANDIDATE"
else:
    deployability_classification = "RESEARCH_ONLY"

final_verdict = {
    "final_strategy_unresolved": False,
    "recommended_candidate": {"model": top_key[0], "K": int(top_key[1]), "allocation_method": top_key[2], "risk_config": top_key[3]},
    "recommended_candidate_net_sharpe": float(top_perf["net_sharpe"]),
    "recommended_candidate_gross_cagr": float(top_perf["gross_cagr"]),
    "recommended_candidate_net_cagr": float(top_perf["net_cagr"]),
    "recommended_candidate_gross_max_drawdown": float(top_perf["gross_max_drawdown"]),
    "spy_cagr": float(spy_core_metrics["cagr"]), "spy_max_drawdown": float(spy_core_metrics["max_drawdown"]),
    "weekly_excess_return_point_estimate": float(top_excess_boot["point_estimate"]),
    "weekly_excess_return_ci_lower": float(top_excess_boot["ci_lower"]),
    "weekly_excess_return_ci_upper": float(top_excess_boot["ci_upper"]),
    "weekly_excess_return_significant": excess_significant,
    "loyo_cagr_range": loyo_cagr_range,
    "break_even_round_trip_bps_vs_spy": top_break_even["break_even_round_trip_bps_vs_spy"],
    "alpha_classification": alpha_classification,
    "deployability_classification": deployability_classification,
}
print(f"[{time.time()-t0:.0f}s] Step 20 (final alpha + deployability classification) done.")
print(json.dumps(final_verdict, indent=2, default=str))

# ---------------------------------------------------------------------------

[42s] Step 20 (final alpha + deployability classification) done.
{
  "final_strategy_unresolved": false,
  "recommended_candidate": {
    "model": "NN_v2",
    "K": 2,
    "allocation_method": "EQUAL_WEIGHT",
    "risk_config": "RISK_A_C_D_E"
  },
  "recommended_candidate_net_sharpe": 0.4061408051432178,
  "recommended_candidate_gross_cagr": 0.11714594607333684,
  "recommended_candidate_net_cagr": 0.06270612836738709,
  "recommended_candidate_gross_max_drawdown": -0.49287252963274153,
  "spy_cagr": 0.0851223169512898,
  "spy_max_drawdown": -0.5571317216698326,
  "weekly_excess_return_point_estimate": 0.0005591903678565867,
  "weekly_excess_return_ci_lower": -0.0003398477362156797,
  "weekly_excess_return_ci_upper": 0.0013849314214584775,
  "weekly_excess_return_significant": false,
  "loyo_cagr_range": 0.041836271916650514,
  "break_even_round_trip_bps_vs_spy": 5.580902099609375,
  "alpha_classification": "ALPHA_POSITIVE_BUT_FRAGILE",
  "deployability_classification": "LIMITED_CAPITAL_

## 21. Sample audit.

In [22]:
# 21. Sample audit.
# ---------------------------------------------------------------------------
sample_audit_rows = []
for model_name in fin.MODELS:
    oof = OOF_BY_MODEL[model_name]
    sample_audit_rows.append({
        "model": model_name, "n_oos_rows": len(oof), "n_weeks": oof["date"].nunique(),
        "first_date": str(oof["date"].min().date()), "last_date": str(oof["date"].max().date()),
        "underlying_auc": float(roc_auc_score(oof.label_binary, oof.predicted_proba)),
        "starting_capital": r10a.STARTING_CAPITAL, "round_trip_cost_bps": ROUND_TRIP_COST_BPS,
        "n_final_candidates": len(CANDIDATES),
    })
final_sample_audit = pd.DataFrame(sample_audit_rows)
print("PART 3 DONE (pre-save).")

# ---------------------------------------------------------------------------

PART 3 DONE (pre-save).


## 22. SPY comparison metrics (gross), all 24 candidates.

In [23]:
# 22. SPY comparison metrics (gross), all 24 candidates.
# ---------------------------------------------------------------------------
vs_spy_rows = []
for key in CANDIDATES:
    model_name, k, alloc, risk_name = key
    excess = weekly_excess_store[key]
    perf = final_performance_summary[(final_performance_summary.model == model_name) & (final_performance_summary.K == k) &
                                      (final_performance_summary.allocation_method == alloc) &
                                      (final_performance_summary.risk_config == risk_name)].iloc[0]
    vs_spy_rows.append({
        "model": model_name, "K": k, "allocation_method": alloc, "risk_config": risk_name,
        "gross_cumulative_return_minus_spy": perf["gross_cumulative_return"] - spy_core_metrics["cumulative_return"],
        "gross_cagr_minus_spy": perf["gross_cagr"] - spy_core_metrics["cagr"],
        "net_cagr_minus_spy": perf["net_cagr"] - spy_core_metrics["cagr"],
        "mean_weekly_excess_return": float(excess.mean()), "excess_return_win_rate": float((excess > 0).mean()),
        "tracking_error": r10a.tracking_error(excess.to_numpy()), "information_ratio": r10a.information_ratio(excess.to_numpy()),
        "gross_max_drawdown_minus_spy": perf["gross_max_drawdown"] - spy_core_metrics["max_drawdown"],
    })
final_vs_spy_metrics = pd.DataFrame(vs_spy_rows)
print(f"[{time.time()-t0:.0f}s] Step 22 (SPY comparison metrics) done.")

# ---------------------------------------------------------------------------

[42s] Step 22 (SPY comparison metrics) done.


## 23. Trade ledger, weekly portfolio returns (gross+net), and equity curves for all 24 candidates + the SPY benchmark.

In [24]:
# 23. Trade ledger, weekly portfolio returns (gross+net), equity curves --
#     all 24 candidates + SPY benchmark row.
# ---------------------------------------------------------------------------
trade_ledger_frames = []
for key in CANDIDATES:
    model_name, k, alloc, risk_name = key
    lg = all_ledgers[key].copy()
    lg.insert(0, "model", model_name); lg.insert(1, "K", k); lg.insert(2, "allocation_method", alloc)
    lg.insert(3, "risk_config", risk_name)
    trade_ledger_frames.append(lg)
final_trade_ledger = pd.concat(trade_ledger_frames, ignore_index=True)

weekly_returns_frames = []
for key in CANDIDATES:
    model_name, k, alloc, risk_name = key
    wg = settlements_gross[key].copy()
    wn = settlements_net[key][["date", "portfolio_return", "equity_after"]].rename(
        columns={"portfolio_return": "net_portfolio_return", "equity_after": "net_equity_after"})
    w = wg.rename(columns={"portfolio_return": "gross_portfolio_return", "equity_after": "gross_equity_after"}).merge(wn, on="date")
    w.insert(0, "model", model_name); w.insert(1, "K", k); w.insert(2, "allocation_method", alloc)
    w.insert(3, "risk_config", risk_name)
    weekly_returns_frames.append(w)
final_weekly_portfolio_returns = pd.concat(weekly_returns_frames, ignore_index=True)

spy_curve_row = spy_weekly_for_metrics.copy()
spy_curve_row.insert(0, "model", "SPY_BENCHMARK"); spy_curve_row.insert(1, "K", np.nan)
spy_curve_row.insert(2, "allocation_method", "N/A"); spy_curve_row.insert(3, "risk_config", "N/A")
spy_curve_row = spy_curve_row.rename(columns={"portfolio_return": "gross_portfolio_return", "equity_after": "gross_equity_after"})
final_equity_curves = pd.concat([final_weekly_portfolio_returns, spy_curve_row], ignore_index=True)
print(f"[{time.time()-t0:.0f}s] Step 23 (ledger/weekly returns/equity curves assembled) done.")

# ---------------------------------------------------------------------------

[42s] Step 23 (ledger/weekly returns/equity curves assembled) done.


## 24. Save all `final_`-prefixed artifacts. None overwrites any prior `r1_`..`r10b_` artifact (disjoint prefix namespace).

In [25]:
# 24. Save all final_ artifacts. Every filename is prefixed final_ -- none
#     overwrites any prior r1_..r10b_ artifact (disjoint prefix namespace).
# ---------------------------------------------------------------------------
final_performance_summary.to_csv(OUTPUTS_ROOT / "final_performance_summary.csv", index=False)
final_break_even_cost.to_csv(OUTPUTS_ROOT / "final_break_even_cost.csv", index=False)
final_cost_robustness_curve.to_csv(OUTPUTS_ROOT / "final_cost_robustness_curve.csv", index=False)
final_alpha_bootstrap_ci.to_csv(OUTPUTS_ROOT / "final_alpha_bootstrap_ci.csv", index=False)
final_yearly_performance.to_csv(OUTPUTS_ROOT / "final_yearly_performance.csv", index=False)
final_stress_period_metrics.to_csv(OUTPUTS_ROOT / "final_stress_period_metrics.csv", index=False)
final_temporal_subperiods.to_csv(OUTPUTS_ROOT / "final_temporal_subperiods.csv", index=False)
final_concentration_by_year.to_csv(OUTPUTS_ROOT / "final_concentration_by_year.csv", index=False)
final_concentration_by_week.to_csv(OUTPUTS_ROOT / "final_concentration_by_week.csv", index=False)
final_concentration_by_etf.to_csv(OUTPUTS_ROOT / "final_concentration_by_etf.csv", index=False)
final_leave_one_year_out.to_csv(OUTPUTS_ROOT / "final_leave_one_year_out.csv", index=False)
final_nn_v1_vs_v2_comparison.to_csv(OUTPUTS_ROOT / "final_nn_v1_vs_v2_comparison.csv", index=False)
final_top2_vs_top3_comparison.to_csv(OUTPUTS_ROOT / "final_top2_vs_top3_comparison.csv", index=False)
final_equal_vs_paper_comparison.to_csv(OUTPUTS_ROOT / "final_equal_vs_paper_comparison.csv", index=False)
final_risk_config_comparison.to_csv(OUTPUTS_ROOT / "final_risk_config_comparison.csv", index=False)
final_rule_b_diagnostic.to_csv(OUTPUTS_ROOT / "final_rule_b_diagnostic.csv", index=False)
final_candidate_ranking.to_csv(OUTPUTS_ROOT / "final_candidate_ranking.csv", index=False)
final_sample_audit.to_csv(OUTPUTS_ROOT / "final_sample_audit.csv", index=False)
final_vs_spy_metrics.to_csv(OUTPUTS_ROOT / "final_vs_spy_metrics.csv", index=False)
final_trade_ledger.to_csv(OUTPUTS_ROOT / "final_trade_ledger.csv", index=False)
final_weekly_portfolio_returns.to_csv(OUTPUTS_ROOT / "final_weekly_portfolio_returns.csv", index=False)
final_equity_curves.to_csv(OUTPUTS_ROOT / "final_equity_curves.csv", index=False)

final_run_summary = {
    "starting_capital": r10a.STARTING_CAPITAL, "round_trip_cost_bps": ROUND_TRIP_COST_BPS,
    "spy_cagr": float(spy_core_metrics["cagr"]), "spy_max_drawdown": float(spy_core_metrics["max_drawdown"]),
    "spy_sharpe": float(spy_core_metrics["sharpe_ratio"]),
    "underlying_auc": {"NN_v1": float(auc_v1), "NN_v2": float(auc_v2)},
    "risk_none_reproduces_r10a": True, "risk_full_reproduces_r10b_abcde": True,
    "n_final_candidates": len(CANDIDATES), "risk_configs_tested": list(fin.RISK_CONFIGS.keys()),
    "final_verdict": final_verdict,
    "runtime_seconds": time.time() - t0,
}
with open(OUTPUTS_ROOT / "final_run_summary.json", "w") as f:
    json.dump(final_run_summary, f, indent=2, default=str)

print(f"[{time.time()-t0:.0f}s] Step 24 (all final_ artifacts saved) done.")
print("ALL DONE.")

artifacts = sorted(p.name for p in OUTPUTS_ROOT.glob("final_*"))
print(f"\n{len(artifacts)} final_ artifacts on disk:")
for a in artifacts:
    print(f"  {a}")

[42s] Step 24 (all final_ artifacts saved) done.
ALL DONE.

23 final_ artifacts on disk:
  final_alpha_bootstrap_ci.csv
  final_break_even_cost.csv
  final_candidate_ranking.csv
  final_concentration_by_etf.csv
  final_concentration_by_week.csv
  final_concentration_by_year.csv
  final_cost_robustness_curve.csv
  final_equal_vs_paper_comparison.csv
  final_equity_curves.csv
  final_leave_one_year_out.csv
  final_nn_v1_vs_v2_comparison.csv
  final_performance_summary.csv
  final_risk_config_comparison.csv
  final_rule_b_diagnostic.csv
  final_run_summary.json
  final_sample_audit.csv
  final_stress_period_metrics.csv
  final_temporal_subperiods.csv
  final_top2_vs_top3_comparison.csv
  final_trade_ledger.csv
  final_vs_spy_metrics.csv
  final_weekly_portfolio_returns.csv
  final_yearly_performance.csv
